# 800 — External Bulk Cohorts

## PREFLIGHT-A — CPTAC-3 cohort eligibility and frozen-input readiness

**Status:** implementation draft; no biological replication analysis has been authorized or executed.

**Objective.** Convert the Notebook 111 CPTAC-3 acquisition audit into auditable, *technical* candidate inventories for prospective external-bulk validation of the three frozen Phase 4 consensus transcriptomic representations.

This initial Notebook 800 implementation is restricted to: (1) referencing the frozen acquisition evidence; (2) auditing primary-tumor candidate coverage and case multiplicity; (3) checking *lexical* overlap against the frozen TCGA sample mapping; (4) inspecting a deterministic, technical STAR Counts gene-ID pilot; and (5) writing non-frozen preflight diagnostics. It does **not** project consensus scores, construct a final analysis cohort, assess cross-dataset replication, test program concordance, or interpret methylation–expression evidence.

**Scientific boundaries.** Cohort eligibility, lineage crosswalk, donor independence, a one-record-per-case rule, full gene coverage, normalization, the replication estimand, null models and multiplicity handling must be prospectively reviewed and frozen before those analyses. Cohort source labels (`primary_site`, `disease_type`) must not be silently converted into TCGA lineages. CPTAC-3 RNA-seq eligibility must not be conditioned on DNA-methylation availability.

**Sources.** `111_cptac3_acquisition_and_audit.ipynb`, `docs/contracts/phase8/NOTEBOOK_800_PREFLIGHT_A_DRAFT.md`, `docs/NOTEBOOK_STYLE_GUIDE.md`, `config/artifact_registry.json` and `config/raw_data_registry.json`.

Run from the project's **Python 3.11.8** Jupyter environment with the local Notebook 111 outputs already generated. No GDC calls, re-download, upstream re-execution or full raw payload MD5 scan is performed.

In [ ]:
# =============================================================================
# Imports
# =============================================================================

import json
import platform

import numpy as np
import pandas as pd

from pancancer_epigenetics.utils.artifact_registry import (
    get_artifact,
    load_artifact_registry,
    resolve_artifact_path,
)
from pancancer_epigenetics.utils.file_checks import calculate_sha256
from pancancer_epigenetics.utils.paths import Paths, project_relative_path
from pancancer_epigenetics.utils.raw_data_registry import load_raw_data_registry

In [ ]:
# =============================================================================
# Notebook 111 handoff and stable Phase 4 input identifiers
# =============================================================================

AUDIT_SUMMARY_PATH = Paths.metadata / "111_cptac3_acquisition_audit_summary.json"
RAW_REGISTRY_PROPOSAL_PATH = Paths.metadata / "111_cptac3_raw_registry_proposal.json"
OUTPUT_DIR = Paths.metadata

AUDIT_INPUT_KEYS = (
    "file_index",
    "candidate_primary",
    "ambiguity_by_lineage",
)

FROZEN_ARTIFACT_IDS = {
    "gene_universe": "phase4.400.cross_system_shared_gene_universe",
    "program_catalog": "phase4.401.consensus_transcriptomic_program_catalog",
    "program_weights": "phase4.401.consensus_transcriptomic_gene_weights",
    "construction_metadata": "phase4.401.consensus_construction_metadata",
    "tcga_sample_mapping": "phase2.203.final_sample_mapping",
}

OUTPUT_PATHS = {
    "lineage_inventory": OUTPUT_DIR / "800_preflight_a_candidate_lineage_inventory.csv",
    "case_multiplicity": OUTPUT_DIR / "800_preflight_a_case_multiplicity.csv",
    "identifier_collisions": OUTPUT_DIR / "800_preflight_a_identifier_collisions.csv",
    "gene_id_pilot": OUTPUT_DIR / "800_preflight_a_gene_id_pilot.csv",
    "readiness": OUTPUT_DIR / "800_preflight_a_readiness.json",
    "rna_adjudication_case_evidence": OUTPUT_DIR / "800_preflight_b_rna_case_evidence.csv",
    "rna_adjudication_file_evidence": OUTPUT_DIR / "800_preflight_b_rna_file_evidence.csv",
    "rna_c_field_coverage": OUTPUT_DIR / "800_preflight_c_rna_field_coverage.csv",
    "rna_c_file_availability": OUTPUT_DIR / "800_preflight_c_rna_file_availability.csv",
    "rna_c_case_coverage": OUTPUT_DIR / "800_preflight_c_rna_case_coverage.csv",
    "rna_d_live_gdc_snapshot": OUTPUT_DIR / "800_preflight_d_gdc_targeted_pilot.json",
    "rna_d_identity": OUTPUT_DIR / "800_preflight_d_rna_pilot_file_identity.csv",
    "rna_d_field_evidence": OUTPUT_DIR / "800_preflight_d_rna_pilot_field_evidence.csv",
    "rna_d_case_summary": OUTPUT_DIR / "800_preflight_d_rna_pilot_case_summary.csv",
    "rna_d_case_field_comparison": OUTPUT_DIR / "800_preflight_d_rna_pilot_case_field_comparison.csv",
}

In [ ]:
# =============================================================================
# Read Notebook 111's observed audit boundary
# =============================================================================

with AUDIT_SUMMARY_PATH.open("r", encoding="utf-8") as handle:
    audit_summary = json.load(handle)

if audit_summary.get("notebook") != 111:
    raise ValueError("The source is not a Notebook 111 audit summary")
if audit_summary.get("cohort_frozen") is not False:
    raise ValueError("Unexpected cohort-freeze flag in Notebook 111 handoff")
if audit_summary.get("phase8_validation_performed") is not False:
    raise ValueError("Unexpected Phase 8 validation status")
if audit_summary.get("n_files_by_modality") != {
    "rnaseq": 2968, "methylation": 2610,
}:
    raise ValueError("CPTAC-3 source-snapshot identity differs from Notebook 111")

print("Notebook 111 source audit: PASS (reported)")
print("Cohort freeze:", audit_summary["cohort_frozen"])
print("External validation performed:", audit_summary["phase8_validation_performed"])

In [ ]:
# =============================================================================
# Resolve and fingerprint the small derived audit tables consumed by 800
# =============================================================================

audit_inputs = {}
for name in AUDIT_INPUT_KEYS:
    record = audit_summary["audit_artifacts"][name]
    path = Paths.root / record["path"]
    if not path.is_file():
        raise FileNotFoundError(f"Missing Notebook 111 derived handoff: {path}")
    if calculate_sha256(path) != record["sha256"]:
        raise ValueError(f"Notebook 111 derived handoff changed: {name}")
    audit_inputs[name] = path

print("Verified Notebook 111 derived handoffs:")
for name, path in audit_inputs.items():
    print(" ", name, project_relative_path(path))

In [ ]:
# =============================================================================
# Read authoritative Notebook 111 tables for new downstream questions only
# =============================================================================

file_index = pd.read_csv(audit_inputs["file_index"], dtype="string")
candidate_primary = pd.read_csv(audit_inputs["candidate_primary"], dtype="string")
upstream_ambiguity = pd.read_csv(
    audit_inputs["ambiguity_by_lineage"], dtype="string"
)

new_use_columns = {
    "modality", "file_id", "filename", "case_uuid", "case_submitter_id",
    "sample_uuid", "aliquot_uuid", "mapping_status",
    "primary_site", "disease_type", "platform",
}
missing_columns = new_use_columns - set(file_index.columns)
if missing_columns:
    raise KeyError(f"Notebook 800 required columns missing: {sorted(missing_columns)}")

# Notebook 111's mapping-status distribution, source totals and primary
# sample-type inventory are authoritative and are not recalculated here.

## Source-registration boundary

The Notebook 111 registry proposal is a review artifact, **not** permission to silently modify the raw-data registry. A missing `cptac3` entry must remain visible as a registration task. Likewise, Phase 4 inputs are frozen and consumed only after exact identity checks; their weights are *not* applied to CPTAC-3 in this preflight.

In [ ]:
# =============================================================================
# Inspect CPTAC-3 raw registry state without mutating it
# =============================================================================

raw_registry = load_raw_data_registry()
with RAW_REGISTRY_PROPOSAL_PATH.open("r", encoding="utf-8") as handle:
    cptac3_registry_proposal = json.load(handle)

cptac3_registered = "cptac3" in raw_registry
print("CPTAC-3 registered:", cptac3_registered)
print("Proposed RNA status:",
      cptac3_registry_proposal["cohorts"]["rnaseq"]["status"])
print("Proposed methylation status:",
      cptac3_registry_proposal["cohorts"]["methylation"]["status"])
if cptac3_registered:
    print("Existing registration requires review against Notebook 111 proposal")

In [ ]:
# =============================================================================
# Verify exact frozen Phase 4 and TCGA reference identities
# =============================================================================

artifact_registry = load_artifact_registry()
frozen_paths = {}
for name, artifact_id in FROZEN_ARTIFACT_IDS.items():
    spec = get_artifact(artifact_registry, artifact_id)
    path = resolve_artifact_path(artifact_registry, artifact_id)
    if spec["status"] != "frozen":
        raise ValueError(f"Required upstream artifact is not frozen: {artifact_id}")
    if path.stat().st_size != spec["size_bytes"]:
        raise ValueError(f"Frozen upstream file size changed: {artifact_id}")
    if calculate_sha256(path) != spec["sha256"]:
        raise ValueError(f"Frozen upstream fingerprint changed: {artifact_id}")
    frozen_paths[name] = path

print("Verified", len(frozen_paths), "frozen input identities")

In [ ]:
# =============================================================================
# Read the frozen gene universe and consensus metadata without scoring
# =============================================================================

shared_genes = pd.read_csv(frozen_paths["gene_universe"], dtype="string")
program_catalog = pd.read_csv(frozen_paths["program_catalog"], dtype="string")
program_weights = pd.read_csv(frozen_paths["program_weights"], dtype="string")
with frozen_paths["construction_metadata"].open("r", encoding="utf-8") as handle:
    construction_metadata = json.load(handle)

required_gene_columns = {"gene_symbol", "tumor_gene_id", "cell_line_gene_id"}
if required_gene_columns - set(shared_genes.columns):
    raise KeyError("Frozen gene universe lacks identifiers required for a technical crosswalk")
if {"consensus_program_id", "gene_symbol", "consensus_weight"} - set(program_weights):
    raise KeyError("Frozen gene weights lack identity columns")

print("Frozen genes:", len(shared_genes))
print("Program IDs:", sorted(program_weights["consensus_program_id"].unique().tolist()))
print("Frozen weight rows:", len(program_weights))
print("Score standardization:", construction_metadata["score_construction"])

## Lineage-aware review of **new** case-level eligibility risks

Notebook 111 already froze technical evidence on sample ancestry, including an
`ambiguity_by_lineage` table with **file-level** denominators and a provisional
primary-tumor inventory. Notebook 800 consumes those audited numbers directly.

The new question is case-level: how many *distinct cases* in each existing
GDC stratum are touched by unresolved file ancestry, and how does that compare
descriptively with the pool of uniquely mapped provisional primary-tumor cases?
These are **not** new primary-tumor calls, new lineage mappings, or eligibility
criteria. The denominator of the unresolved-case fraction contains **all sample
types** from the acquired files, and cannot be interpreted as primary-tumor
attrition. A dedicated primary-compatible unresolved-sample sensitivity, and
the final one-case selection rule, remain separate decisions.

In [ ]:
# =============================================================================
# Reuse Notebook 111 source-stratum counts; compute only new case-level risk
# =============================================================================

STRATUM_KEYS = ["modality", "primary_site", "disease_type", "platform"]
UPSTREAM_COUNT_COLUMNS = [
    "n_files", "n_cases", "n_ambiguous_files", "ambiguous_file_fraction",
]
missing_upstream = (
    set(STRATUM_KEYS + UPSTREAM_COUNT_COLUMNS) - set(upstream_ambiguity.columns)
)
if missing_upstream:
    raise KeyError(f"111 lineage audit missing fields: {sorted(missing_upstream)}")

source_counts = upstream_ambiguity[
    STRATUM_KEYS + UPSTREAM_COUNT_COLUMNS
].copy()
for column in UPSTREAM_COUNT_COLUMNS:
    source_counts[column] = pd.to_numeric(
        source_counts[column], errors="raise"
    )

unresolved_case_counts = (
    file_index.loc[file_index["mapping_status"].eq("ambiguous_sample_ancestry")]
    .groupby(STRATUM_KEYS, dropna=False)["case_uuid"]
    .nunique()
    .rename("cases_with_ambiguous_files")
    .reset_index()
)
source_counts = source_counts.merge(
    unresolved_case_counts, on=STRATUM_KEYS, how="left", validate="one_to_one"
)
source_counts["cases_with_ambiguous_files"] = (
    source_counts["cases_with_ambiguous_files"].fillna(0).astype("int64")
)
source_counts["fraction_all_cases_with_ambiguous_files"] = (
    source_counts["cases_with_ambiguous_files"] / source_counts["n_cases"]
)

In [ ]:
# =============================================================================
# Contextualize new case-level ambiguity against the 111 primary inventory
# =============================================================================

# Notebook 111 reports primary counts by source site/disease; this new
# platform-specific case view is descriptive only and is not an eligibility gate.
primary_case_counts = (
    candidate_primary.groupby(STRATUM_KEYS, dropna=False)["case_uuid"]
    .nunique()
    .rename("provisional_primary_cases")
    .reset_index()
)

lineage_inventory = source_counts.merge(
    primary_case_counts, on=STRATUM_KEYS, how="left", validate="one_to_one"
)
lineage_inventory["provisional_primary_cases"] = (
    lineage_inventory["provisional_primary_cases"].fillna(0).astype("int64")
)
lineage_inventory = lineage_inventory.sort_values(
    ["modality", "cases_with_ambiguous_files", "primary_site", "disease_type"],
    ascending=[True, False, True, True], kind="stable"
).reset_index(drop=True)

display(lineage_inventory[
    ["modality", "primary_site", "disease_type", "platform",
     "n_cases", "cases_with_ambiguous_files",
     "fraction_all_cases_with_ambiguous_files", "provisional_primary_cases"]
].head(25))
print("Lineage crosswalk and primary-specific attrition: NOT ASSESSED")

In [ ]:
# =============================================================================
# Quantify case-level multiplicity in provisional primary records
# =============================================================================

case_multiplicity = (
    candidate_primary.groupby(["modality", "case_uuid"], dropna=False)
    .agg(
        n_files=("file_id", "nunique"),
        n_samples=("sample_uuid", "nunique"),
        n_aliquots=("aliquot_uuid", "nunique"),
        n_platforms=("platform", "nunique"),
        file_uuid_values=("file_id", lambda v: "|".join(sorted(set(v.dropna())))),
        sample_uuid_values=("sample_uuid", lambda v: "|".join(sorted(set(v.dropna())))),
        aliquot_uuid_values=("aliquot_uuid", lambda v: "|".join(sorted(set(v.dropna())))),
        primary_site_values=("primary_site", lambda v: "|".join(sorted(set(v.dropna())))),
        disease_type_values=("disease_type", lambda v: "|".join(sorted(set(v.dropna())))),
    )
    .reset_index()
)

case_multiplicity["needs_selection_policy"] = (
    case_multiplicity[["n_files", "n_samples", "n_aliquots"]].gt(1).any(axis=1)
)
case_multiplicity["selected_file_id"] = pd.Series(
    pd.NA, index=case_multiplicity.index, dtype="string"
)
case_multiplicity["selection_status"] = "NOT_FROZEN"
case_multiplicity = case_multiplicity.sort_values(
    ["modality", "case_uuid"], kind="stable"
).reset_index(drop=True)

display(
    case_multiplicity.groupby("modality")
    .agg(
        candidate_cases=("case_uuid", "nunique"),
        cases_needing_review=("needs_selection_policy", "sum"),
        maximum_files_per_case=("n_files", "max"),
        maximum_samples_per_case=("n_samples", "max"),
    )
)
print("No case-level selection, averaging or technical-replicate pooling performed")

## RNA-seq per-case multiplicity: outcome-blind structural classification

The initial 800 preflight identified **288 provisional RNA-seq cases requiring multiplicity review**. The next technical task is to distinguish case-level identifier structures **without** reopening Notebook 111's file-to-sample ancestry audit or making a biological sample-selection decision.

Classification is based only on already audited file, sample, and aliquot UUIDs. **More than one file on a single aliquot is not, by itself, proof of technical replication or equivalence of processed files.** More than one aliquot under one sample is not automatically exchangeable, and distinct sample UUIDs under one case cannot be pooled as independent patients.

Cases with internally inconsistent counts or cross-file mapping conflicts must remain explicitly unresolved. Even a case with one file is merely *technically simple*, **not** finally eligible: donor independence, lineage harmonization, technical QC and prospective Phase 8 scoring policy remain open.

In [ ]:
# =============================================================================
# Classify RNA-seq case multiplicity from the existing 800 case inventory
# =============================================================================

rna_case_mask = case_multiplicity["modality"].eq("rnaseq")
rna_cases = case_multiplicity.loc[rna_case_mask].copy()

inconsistent_counts = (
    rna_cases[["n_files", "n_samples", "n_aliquots"]].le(0).any(axis=1)
    | rna_cases["n_aliquots"].gt(rna_cases["n_files"])
    | rna_cases["n_samples"].gt(rna_cases["n_aliquots"])
)
rna_cases["rna_multiplicity_class"] = np.select(
    [
        inconsistent_counts,
        rna_cases["n_samples"].gt(1),
        rna_cases["n_aliquots"].gt(1),
        rna_cases["n_files"].gt(1),
    ],
    [
        "INCONSISTENT_BIOSPECIMEN_COUNTS",
        "MULTIPLE_SAMPLES_IN_CASE",
        "ONE_SAMPLE_MULTIPLE_ALIQUOTS",
        "ONE_ALIQUOT_MULTIPLE_FILES",
    ],
    default="SINGLE_FILE_SAMPLE_ALIQUOT",
)
print("RNA-seq provisional cases classified:", len(rna_cases))
print("No record selected, pooled, excluded or scored")

In [ ]:
# =============================================================================
# Screen for cross-file ancestry conflicts not resolved by single-file links
# =============================================================================

rna_candidate_rows = candidate_primary.loc[
    candidate_primary["modality"].eq("rnaseq"),
    ["case_uuid", "sample_uuid", "aliquot_uuid", "file_id"],
].copy()

aliquot_links = (
    rna_candidate_rows.groupby("aliquot_uuid")
    .agg(
        n_parent_samples=("sample_uuid", "nunique"),
        n_parent_cases=("case_uuid", "nunique"),
    )
)
sample_links = (
    rna_candidate_rows.groupby("sample_uuid")
    .agg(n_parent_cases=("case_uuid", "nunique"))
)
conflicting_aliquots = set(aliquot_links.loc[
    aliquot_links["n_parent_samples"].gt(1)
    | aliquot_links["n_parent_cases"].gt(1)
].index)
conflicting_samples = set(sample_links.loc[
    sample_links["n_parent_cases"].gt(1)
].index)

rna_cases["cross_file_ancestry_conflict"] = (
    rna_cases["aliquot_uuid_values"].map(
        lambda s: any(part in conflicting_aliquots for part in s.split("|"))
    )
    | rna_cases["sample_uuid_values"].map(
        lambda s: any(part in conflicting_samples for part in s.split("|"))
    )
)
rna_cases.loc[
    rna_cases["cross_file_ancestry_conflict"],
    "rna_multiplicity_class",
] = "CROSS_FILE_BIOSPECIMEN_CONFLICT"

print("Aliquot UUIDs with incompatible cross-file parents:", len(conflicting_aliquots))
print("Sample UUIDs with incompatible cross-file cases:", len(conflicting_samples))
print("Affected RNA cases:", int(rna_cases["cross_file_ancestry_conflict"].sum()))

In [ ]:
# =============================================================================
# Show nonoverlapping RNA-seq structural classes and review candidates
# =============================================================================

rna_class_summary = (
    rna_cases.groupby("rna_multiplicity_class", dropna=False)
    .agg(
        n_cases=("case_uuid", "nunique"),
        max_files_in_case=("n_files", "max"),
        max_samples_in_case=("n_samples", "max"),
    )
    .reset_index()
    .sort_values(["n_cases", "rna_multiplicity_class"], ascending=[False, True])
)
rna_class_summary["fraction_provisional_rna_cases"] = (
    rna_class_summary["n_cases"] / len(rna_cases)
)

display(rna_class_summary)
print(
    "Cases previously flagged for multiplicity review:",
    int(rna_cases["needs_selection_policy"].sum()),
)
print("Unresolved case selection: ALL — no file selection rule frozen")

# Preserve exact source labels; mixed labels must remain visible.
display(
    rna_cases.loc[rna_cases["needs_selection_policy"]]
    .groupby(["primary_site_values", "rna_multiplicity_class"], dropna=False)
    .size().rename("n_cases").reset_index()
    .sort_values(["n_cases", "primary_site_values"], ascending=[False, True])
    .head(20)
)

In [ ]:
# =============================================================================
# Annotate the existing case-level output; create no duplicate source inventory
# =============================================================================

case_multiplicity["rna_multiplicity_class"] = pd.Series(
    pd.NA, index=case_multiplicity.index, dtype="string"
)
case_multiplicity["rna_cross_file_ancestry_conflict"] = pd.Series(
    pd.NA, index=case_multiplicity.index, dtype="boolean"
)

case_multiplicity.loc[
    rna_case_mask, "rna_multiplicity_class"
] = rna_cases["rna_multiplicity_class"].to_numpy()
case_multiplicity.loc[
    rna_case_mask, "rna_cross_file_ancestry_conflict"
] = rna_cases["cross_file_ancestry_conflict"].to_numpy()

if (
    rna_class_summary["n_cases"].sum() != len(rna_cases)
    or case_multiplicity.loc[rna_case_mask, "rna_multiplicity_class"].isna().any()
):
    raise ValueError("Incomplete RNA-seq case multiplicity classification")

print("Annotated output:", project_relative_path(OUTPUT_PATHS["case_multiplicity"]))
print("Classification: descriptive only; no sample selected")

## PREFLIGHT-B — Outcome-blind RNA-seq multiplicity adjudication evidence

**Question:** for the 288 cases containing multiple uniquely mapped provisional primary-tumor RNA-seq files/samples/aliquots, which *existing, audited* source metadata can distinguish candidate biospecimens? Does that information supply a defensible basis for selecting one observation per case?

This is a **metadata evidence audit**, not an actual adjudication. It consumes the already classified RNA cases and Notebook 111's frozen candidate file index; it does not rerun source acquisition, reopen ancestral assignment, reinterpret ambiguous files, or inspect any gene expression or program scores.

The preserved GDC descriptors `sample_type`, `tissue_type` and `tumor_descriptor` may identify disagreements but cannot resolve tumor-region/timepoint differences by themselves. Sample/aliquot submitter IDs are identifiers, not evidence of relative quality. Provider file size, MD5 and generic STAR workflow are **not sequencing QC selection metrics**. Even identical submitted labels do not prove samples are interchangeable.

All 288 case/file identities remain in a **NOT_FROZEN** state. Any future one-per-case decision must use a separately reviewed outcome-blind policy, with enough objective specimen provenance and technical QC. Cases without a defensible selection basis remain unresolved; never select on UUID ordering, file size, gene expression or program score.

In [ ]:
# =============================================================================
# Restrict new evidence review to the 288 RNA cases already identified in 800
# =============================================================================

rna_review_cases = rna_cases.loc[
    rna_cases["needs_selection_policy"], [
        "case_uuid", "rna_multiplicity_class", "n_files", "n_samples",
        "n_aliquots", "cross_file_ancestry_conflict",
    ]
].copy()

RNA_SOURCE_EVIDENCE_COLS = [
    "case_uuid", "file_id", "filename", "sample_uuid", "sample_submitter_id",
    "aliquot_uuid", "aliquot_submitter_id", "sample_type", "tissue_type",
    "tumor_descriptor", "primary_site", "disease_type", "project_id",
    "workflow_type", "platform", "size_bytes", "provider_md5",
]
missing = set(RNA_SOURCE_EVIDENCE_COLS) - set(candidate_primary.columns)
if missing:
    raise KeyError("Required existing Notebook 111 metadata absent: " + str(sorted(missing)))

rna_adjudication_files = candidate_primary.loc[
    candidate_primary["modality"].eq("rnaseq")
    & candidate_primary["case_uuid"].isin(rna_review_cases["case_uuid"]),
    RNA_SOURCE_EVIDENCE_COLS,
].copy()

if (
    rna_review_cases["case_uuid"].duplicated().any()
    or rna_adjudication_files["file_id"].duplicated().any()
    or set(rna_adjudication_files["case_uuid"]) != set(rna_review_cases["case_uuid"])
    or len(rna_adjudication_files) != int(rna_review_cases["n_files"].sum())
):
    raise ValueError("800-B candidate review does not reconcile to 800-A case inventory")

print("Cases under metadata adjudication review:", len(rna_review_cases))
print("Existing primary RNA file records under review:", len(rna_adjudication_files))

In [ ]:
# =============================================================================
# Quantify per-case specimen descriptors and source-file metadata coverage
# =============================================================================

def nonnull_count(series):
    return int(series.notna().sum())

def unique_nonnull_count(series):
    return int(series.nunique(dropna=True))

evidence = (
    rna_adjudication_files.groupby("case_uuid", dropna=False)
    .agg(
        observed_files=("file_id", "nunique"),
        observed_samples=("sample_uuid", "nunique"),
        observed_aliquots=("aliquot_uuid", "nunique"),
        sample_submitter_ids=("sample_submitter_id", unique_nonnull_count),
        sample_submitter_id_file_coverage=("sample_submitter_id", nonnull_count),
        aliquot_submitter_ids=("aliquot_submitter_id", unique_nonnull_count),
        aliquot_submitter_id_file_coverage=("aliquot_submitter_id", nonnull_count),
        tissue_type_values=("tissue_type", unique_nonnull_count),
        tissue_type_file_coverage=("tissue_type", nonnull_count),
        tumor_descriptor_values=("tumor_descriptor", unique_nonnull_count),
        tumor_descriptor_file_coverage=("tumor_descriptor", nonnull_count),
        source_site_values=("primary_site", unique_nonnull_count),
        source_disease_values=("disease_type", unique_nonnull_count),
        project_values=("project_id", unique_nonnull_count),
        workflow_values=("workflow_type", unique_nonnull_count),
        platform_values=("platform", unique_nonnull_count),
        distinct_provider_md5=("provider_md5", unique_nonnull_count),
    ).reset_index()
)
rna_adjudication_cases = rna_review_cases.merge(
    evidence, on="case_uuid", how="left", validate="one_to_one"
)

if not (
    rna_adjudication_cases["n_files"].eq(rna_adjudication_cases["observed_files"]).all()
    and rna_adjudication_cases["n_samples"].eq(
        rna_adjudication_cases["observed_samples"]
    ).all()
    and rna_adjudication_cases["n_aliquots"].eq(
        rna_adjudication_cases["observed_aliquots"]
    ).all()
):
    raise ValueError("New specimen evidence does not match prior case structure")

display(
    rna_adjudication_cases[[
        "rna_multiplicity_class", "observed_files",
        "tissue_type_file_coverage", "tumor_descriptor_file_coverage",
        "sample_submitter_id_file_coverage",
        "aliquot_submitter_id_file_coverage",
    ]].groupby("rna_multiplicity_class").sum()
)

In [ ]:
# =============================================================================
# Flag metadata discordance and insufficiency, without resolving samples
# =============================================================================

rna_adjudication_cases["source_descriptor_discordance"] = (
    rna_adjudication_cases[[
        "tissue_type_values", "tumor_descriptor_values",
        "source_site_values", "source_disease_values",
        "project_values", "workflow_values", "platform_values",
    ]].gt(1).any(axis=1)
)
rna_adjudication_cases["source_descriptor_incomplete"] = (
    rna_adjudication_cases["tissue_type_file_coverage"].lt(
        rna_adjudication_cases["n_files"]
    )
    | rna_adjudication_cases["tumor_descriptor_file_coverage"].lt(
        rna_adjudication_cases["n_files"]
    )
    | rna_adjudication_cases["sample_submitter_id_file_coverage"].lt(
        rna_adjudication_cases["n_files"]
    )
    | rna_adjudication_cases["aliquot_submitter_id_file_coverage"].lt(
        rna_adjudication_cases["n_files"]
    )
)
rna_adjudication_cases["same_md5_across_distinct_files"] = (
    rna_adjudication_cases["distinct_provider_md5"].lt(
        rna_adjudication_cases["n_files"]
    )
)

rna_adjudication_cases["adjudication_evidence_class"] = np.select(
    [
        rna_adjudication_cases["cross_file_ancestry_conflict"],
        rna_adjudication_cases["source_descriptor_discordance"],
        rna_adjudication_cases["source_descriptor_incomplete"],
        rna_adjudication_cases["rna_multiplicity_class"].eq(
            "MULTIPLE_SAMPLES_IN_CASE"
        ),
    ],
    [
        "CROSS_FILE_CONFLICT_UNRESOLVED",
        "SOURCE_DESCRIPTORS_DISCORDANT_REVIEW",
        "SOURCE_DESCRIPTORS_INCOMPLETE_REVIEW",
        "MULTIPLE_SAMPLES_NO_QC_SELECTION_BASIS",
    ],
    default="MULTIPLE_ALIQUOTS_OR_FILES_NO_QC_SELECTION_BASIS",
)
rna_adjudication_cases["selection_evidence_status"] = (
    "INSUFFICIENT_TO_SELECT_WITHOUT_PROSPECTIVE_QC_POLICY"
)
rna_adjudication_cases["selected_file_id"] = pd.Series(
    pd.NA, index=rna_adjudication_cases.index, dtype="string"
)

display(
    rna_adjudication_cases["adjudication_evidence_class"]
    .value_counts(dropna=False)
    .rename_axis("evidence_class").reset_index(name="n_cases")
)
print("No files selected or excluded; no technical QC read from STAR payloads")

In [ ]:
# =============================================================================
# Describe the evidence limits by source tumor context without rerunning 111
# =============================================================================

rna_case_context = rna_cases.loc[
    rna_cases["needs_selection_policy"],
    ["case_uuid", "primary_site_values", "disease_type_values"],
].copy()
rna_adjudication_cases = rna_adjudication_cases.merge(
    rna_case_context, on="case_uuid", validate="one_to_one"
)
display(
    rna_adjudication_cases.groupby(
        ["primary_site_values", "rna_multiplicity_class"], dropna=False
    ).agg(
        n_cases=("case_uuid", "nunique"),
        descriptor_discordance=("source_descriptor_discordance", "sum"),
        incomplete_descriptors=("source_descriptor_incomplete", "sum"),
        same_md5_file_groups=("same_md5_across_distinct_files", "sum"),
    )
    .reset_index()
    .sort_values(["n_cases", "primary_site_values"], ascending=[False, True])
    .head(25)
)
print(
    "Cases with source-descriptor disagreement:",
    int(rna_adjudication_cases["source_descriptor_discordance"].sum())
)
print(
    "Cases with at least one missing specimen descriptor field:",
    int(rna_adjudication_cases["source_descriptor_incomplete"].sum())
)
print("True technical replicate identity / relative RNA-seq QC: NOT ASSESSED")

In [ ]:
# =============================================================================
# Prepare auditable per-file metadata evidence; preserve case status UNRESOLVED
# =============================================================================

rna_adjudication_files = rna_adjudication_files.merge(
    rna_adjudication_cases[
        ["case_uuid", "rna_multiplicity_class", "adjudication_evidence_class",
         "selection_evidence_status"]
    ],
    on="case_uuid", validate="many_to_one"
)
rna_adjudication_files["candidate_selected"] = False
rna_adjudication_files["selection_performed"] = False

rna_adjudication_cases = rna_adjudication_cases.sort_values(
    "case_uuid", kind="stable"
).reset_index(drop=True)
rna_adjudication_files = rna_adjudication_files.sort_values(
    ["case_uuid", "sample_uuid", "aliquot_uuid", "file_id"], kind="stable"
).reset_index(drop=True)

if (
    len(rna_adjudication_cases) != int(rna_cases["needs_selection_policy"].sum())
    or rna_adjudication_cases["selected_file_id"].notna().any()
    or rna_adjudication_files["candidate_selected"].any()
):
    raise ValueError("Evidence-only 800-B unexpectedly selected or lost cases")

print("Case-level adjudication evidence rows:", len(rna_adjudication_cases))
print("File-level adjudication evidence rows:", len(rna_adjudication_files))
print("Selection eligibility rule: NOT_FROZEN")

## PREFLIGHT-C — Source specimen provenance and technical QC *availability* (not adjudication)

**Question.** Does the **preserved acquisition-time GDC RNA metadata snapshot** contain sufficiently specific collection, spatial, preparation or comparable QC information to distinguish the already identified 649 RNA files from the 288 cases requiring review?

This stage first fingerprints the exact metadata snapshot already reported by Notebook 111 and examines **only those 649 audited file IDs**. For each file, additional fields are read **only from the case → sample → portion → analyte → aliquot path matching the exact three UUIDs already resolved in Notebook 111**. The step does not redo ancestry resolution or rescue the ambiguous records.

The audit distinguishes **key present**, **informative scalar value** (excluding common unknown/not-reported placeholders), and **multiple different values on matching hierarchical paths**. A GDC file creation date is **not** a specimen collection date. A sample submitter ID, portion number or concentration is **not** a QC score; generic `sample_type`, `tissue_type` or `tumor_descriptor` does **not** establish a tumor sampling region. An apparently uniform field does not prove identical tumor regions/timepoints.

**QC scope:** structural *availability of metadata fields* only. No processed STAR expression payloads, source QC reports, alignment logs or newly queried GDC information are inspected. Absent keys mean **not present in this audited snapshot**, *not* absent from the provider. No sample is selected or excluded; RNA file quality and relative quality remain unresolved.

In [ ]:
# =============================================================================
# Read only the RNA acquisition-time metadata snapshot already frozen by 111
# =============================================================================

RNA_GDC_METADATA_PATH = (
    Paths.config / "manifests" / "cptac3" / "gdc_metadata_cptac3_rnaseq.json"
)
expected_rna_metadata_sha = audit_summary[
    "source_fingerprints"
]["rnaseq"]["metadata"]["sha256"]
if calculate_sha256(RNA_GDC_METADATA_PATH) != expected_rna_metadata_sha:
    raise ValueError("RNA acquisition-time GDC metadata differs from Notebook 111")

with RNA_GDC_METADATA_PATH.open("r", encoding="utf-8") as handle:
    rna_snapshot = json.load(handle)
if not isinstance(rna_snapshot, dict) or not isinstance(
    rna_snapshot.get("data"), dict
):
    raise ValueError("Unexpected RNA metadata snapshot envelope")
rna_metadata_hits = rna_snapshot["data"].get("hits")
if not isinstance(rna_metadata_hits, list):
    raise ValueError("RNA metadata snapshot has no data.hits list")

# The snapshot is already source-audited by 111: index only 800-B's 649 files.
review_file_ids = set(rna_adjudication_files["file_id"])
rna_c_hits = {}
for hit in rna_metadata_hits:
    file_uuid = hit.get("file_id") or hit.get("id")
    if file_uuid in review_file_ids:
        if file_uuid in rna_c_hits:
            raise ValueError("Duplicate file ID in reviewed RNA snapshot subset")
        rna_c_hits[file_uuid] = hit
if set(rna_c_hits) != review_file_ids:
    raise ValueError("Missing 800-B file IDs in frozen RNA metadata snapshot")

print("Exact reviewed GDC metadata files:", len(rna_c_hits))
print("RNA metadata snapshot SHA-256: PASS (Notebook 111 reference)")
print("No API query, source re-acquisition or STAR payload QC scan")

In [ ]:
# =============================================================================
# Locate exact already-audited triplets to expose additional source fields
# =============================================================================

def exact_source_nodes(hit, case_uuid, sample_uuid, aliquot_uuid):
    paths = []
    for case in hit.get("cases") or []:
        if case.get("case_id") != case_uuid:
            continue
        for sample in case.get("samples") or []:
            if sample.get("sample_id") != sample_uuid:
                continue
            for portion in sample.get("portions") or []:
                for analyte in portion.get("analytes") or []:
                    for aliquot in analyte.get("aliquots") or []:
                        if aliquot.get("aliquot_id") == aliquot_uuid:
                            paths.append({
                                "sample": sample,
                                "portion": portion,
                                "analyte": analyte,
                                "aliquot": aliquot,
                            })
    return paths

rna_c_node_maps = {}
rna_c_path_rows = []
for row in rna_adjudication_files.itertuples(index=False):
    hit = rna_c_hits[row.file_id]
    paths = exact_source_nodes(
        hit, row.case_uuid, row.sample_uuid, row.aliquot_uuid
    )
    if not paths:
        raise ValueError(
            "Notebook 111 audited case/sample/aliquot path not found for "
            + str(row.file_id)
        )
    rna_c_node_maps[row.file_id] = {
        "file": [hit],
        "analysis": [hit.get("analysis")] if isinstance(
            hit.get("analysis"), dict
        ) else [],
        **{
            scope: [entry[scope] for entry in paths]
            for scope in ("sample", "portion", "analyte", "aliquot")
        },
    }
    rna_c_path_rows.append({
        "file_id": row.file_id,
        "case_uuid": row.case_uuid,
        "sample_uuid": row.sample_uuid,
        "aliquot_uuid": row.aliquot_uuid,
        "exact_hierarchy_path_rows": len(paths),
    })

rna_c_exact_path_index = pd.DataFrame(rna_c_path_rows)
print("Exact previously audited triplets represented:",
      len(rna_c_exact_path_index))
print("Files with >1 exact hierarchy row (not distinct parents):",
      int(rna_c_exact_path_index["exact_hierarchy_path_rows"].gt(1).sum()))

In [ ]:
# =============================================================================
# Define conservative, testable source metadata availability domains
# =============================================================================

RNA_C_FIELDS = {
    "generic_sample_descriptors": [
        ("sample", "sample_type"), ("sample", "tissue_type"),
        ("sample", "tumor_descriptor"),
    ],
    "collection_timing": [
        ("sample", "days_to_collection"),
        ("sample", "days_to_sample_procurement"),
        ("sample", "time_between_clamping_and_freezing"),
        ("sample", "time_between_excision_and_freezing"),
    ],
    "sampling_location_detail": [
        ("sample", "tumor_region"), ("sample", "tumor_subsite"),
        ("sample", "sample_collection_site"), ("sample", "anatomic_site"),
        ("sample", "specimen_laterality"),
    ],
    "processing_provenance": [
        ("sample", "method_of_sample_procurement"),
        ("sample", "preservation_method"), ("sample", "is_ffpe"),
        ("portion", "is_ffpe"), ("portion", "portion_number"),
        ("portion", "weight"), ("analyte", "analyte_type"),
        ("analyte", "concentration"), ("aliquot", "concentration"),
    ],
    "identifiers_not_qc": [
        ("sample", "submitter_id"), ("portion", "submitter_id"),
        ("analyte", "submitter_id"), ("aliquot", "submitter_id"),
    ],
    "file_processing_dates_not_collection": [
        ("file", "created_datetime"), ("file", "updated_datetime"),
        ("analysis", "analysis_id"), ("analysis", "workflow_type"),
        ("analysis", "workflow_version"),
    ],
    "quantitative_qc_metadata_probes": [
        ("file", "total_reads"), ("file", "mapped_reads"),
        ("file", "average_read_length"), ("file", "average_base_quality"),
        ("analysis", "total_reads"), ("analysis", "mapped_reads"),
        ("analysis", "mapping_rate"), ("analysis", "duplication_rate"),
        ("analysis", "percent_aligned_reads"),
    ],
}

# Presence here means present in the preserved projection, not in all GDC data.
PLACEHOLDER_STRINGS = {
    "", "unknown", "not reported", "not available", "not applicable",
    "n/a", "na", "null", "none", "unspecified", "--",
}

def informative_scalar(value):
    if value is None or isinstance(value, (list, tuple, dict)):
        return False
    if isinstance(value, str):
        return value.strip().casefold() not in PLACEHOLDER_STRINGS
    if isinstance(value, (bool, int, float)):
        return bool(pd.notna(value))
    return False

def source_value_status(nodes, key):
    values = [
        item[key] for item in nodes
        if isinstance(item, dict) and key in item
    ]
    useful = {str(v).strip() for v in values if informative_scalar(v)}
    return bool(values), bool(useful), len(useful) > 1

all_probes = {
    (scope, key)
    for group in RNA_C_FIELDS.values()
    for scope, key in group
}
print("Explicit availability probes:", len(all_probes))
print("No thresholds for donor identity, specimen equivalence or quality")

In [ ]:
# =============================================================================
# Inventory observed scalar snapshot keys and quantitative-QC probe coverage
# =============================================================================

observed_scalar_keys = set()
for node_map in rna_c_node_maps.values():
    for scope, nodes in node_map.items():
        for node in nodes:
            if isinstance(node, dict):
                observed_scalar_keys.update(
                    (scope, key) for key, value in node.items()
                    if not isinstance(value, (dict, list, tuple))
                )

# Inventory includes actual fields, plus explicit absent probes for auditability.
all_field_keys = sorted(all_probes | observed_scalar_keys)
rna_c_field_records = []
for scope, key in all_field_keys:
    n_key, n_informative, n_discordant = 0, 0, 0
    for nodes in rna_c_node_maps.values():
        present, informative, discordant = source_value_status(
            nodes[scope], key
        )
        n_key += int(present)
        n_informative += int(informative)
        n_discordant += int(discordant)
    assigned_domains = sorted(
        name for name, fields in RNA_C_FIELDS.items()
        if (scope, key) in fields
    )
    rna_c_field_records.append({
        "scope": scope, "field": key,
        "planned_domains": "|".join(assigned_domains),
        "observed_scalar_key": (scope, key) in observed_scalar_keys,
        "n_files_with_key": n_key,
        "n_files_with_informative_scalar": n_informative,
        "n_files_with_multiple_informative_values": n_discordant,
        "n_reviewed_files": len(rna_c_node_maps),
    })
rna_c_field_coverage = pd.DataFrame(rna_c_field_records)
display(
    rna_c_field_coverage.loc[
        rna_c_field_coverage["planned_domains"].ne("")
    ].sort_values(["planned_domains", "scope", "field"])
)
print(
    "Observed scalar fields not in explicit probe list:",
    len(observed_scalar_keys - all_probes)
)

In [ ]:
# =============================================================================
# Flag usable specimen/QC metadata *availability* for each of 649 files
# =============================================================================

rna_c_file_rows = []
for row in rna_adjudication_files.itertuples(index=False):
    nodes = rna_c_node_maps[row.file_id]
    new_record = {
        "file_id": row.file_id,
        "case_uuid": row.case_uuid,
        "sample_uuid": row.sample_uuid,
        "aliquot_uuid": row.aliquot_uuid,
        "rna_multiplicity_class": row.rna_multiplicity_class,
    }
    for domain, probes in RNA_C_FIELDS.items():
        statuses = [
            source_value_status(nodes[scope], key)
            for scope, key in probes
        ]
        new_record["informative_" + domain] = any(
            informative for _, informative, _ in statuses
        )
        new_record["discordant_" + domain] = any(
            discordant for _, _, discordant in statuses
        )
        new_record["n_informative_fields_" + domain] = sum(
            informative for _, informative, _ in statuses
        )
    rna_c_file_rows.append(new_record)

rna_c_file_availability = pd.DataFrame(rna_c_file_rows).merge(
    rna_c_exact_path_index[["file_id", "exact_hierarchy_path_rows"]],
    on="file_id", validate="one_to_one"
).sort_values(["case_uuid", "sample_uuid", "aliquot_uuid", "file_id"])
rna_c_file_availability = rna_c_file_availability.reset_index(drop=True)

summary_fields = [
    "informative_" + name for name in RNA_C_FIELDS
]
display(
    rna_c_file_availability.groupby("rna_multiplicity_class")[
        summary_fields
    ].sum()
)
print("Non-informative placeholders are not counted as usable values")
print("Provider-level and processed STAR QC: NOT TESTED")

In [ ]:
# =============================================================================
# Summarize 288-case coverage without adjudicating within-case candidates
# =============================================================================

rna_c_case_coverage = (
    rna_c_file_availability.groupby("case_uuid")
    .agg(
        n_files=("file_id", "nunique"),
        n_exact_hierarchy_rows=("exact_hierarchy_path_rows", "sum"),
        **{
            "n_files_informative_" + name: (
                "informative_" + name, "sum"
            )
            for name in RNA_C_FIELDS
        },
        **{
            "n_files_discordant_" + name: (
                "discordant_" + name, "sum"
            )
            for name in RNA_C_FIELDS
        },
    )
    .reset_index()
    .merge(
        rna_review_cases[[
            "case_uuid", "rna_multiplicity_class", "n_files"
        ]].rename(columns={"n_files": "audited_n_files"}),
        on="case_uuid", validate="one_to_one"
    )
)
if (
    len(rna_c_case_coverage) != len(rna_review_cases)
    or not rna_c_case_coverage["n_files"].eq(
        rna_c_case_coverage["audited_n_files"]
    ).all()
):
    raise ValueError("800-C changed the source-snapshot review denominator")
rna_c_case_coverage["selection_policy_status"] = "NOT_FROZEN"
rna_c_case_coverage = rna_c_case_coverage.sort_values(
    "case_uuid", kind="stable"
).reset_index(drop=True)

display(
    rna_c_case_coverage.groupby("rna_multiplicity_class")
    .agg(
        cases=("case_uuid", "nunique"),
        with_any_collection_timing=(
            "n_files_informative_collection_timing", lambda v: int(v.gt(0).sum())
        ),
        with_all_collection_timing=(
            "n_files_informative_collection_timing",
            lambda v: int(v.eq(
                rna_c_case_coverage.loc[v.index, "n_files"]
            ).sum()),
        ),
        with_any_location_detail=(
            "n_files_informative_sampling_location_detail",
            lambda v: int(v.gt(0).sum())
        ),
        with_any_quantitative_qc_metadata=(
            "n_files_informative_quantitative_qc_metadata_probes",
            lambda v: int(v.gt(0).sum())
        ),
    )
)
print("All 288 cases remain unresolved for one-record-per-case selection")

In [ ]:
# =============================================================================
# Enforce no-selection and scope guarantees for this availability-only stage
# =============================================================================

if (
    len(rna_c_file_availability) != len(rna_adjudication_files)
    or rna_c_file_availability["file_id"].duplicated().any()
    or rna_c_case_coverage["case_uuid"].duplicated().any()
    or rna_adjudication_files["candidate_selected"].any()
    or rna_adjudication_cases["selected_file_id"].notna().any()
    or rna_c_case_coverage["selection_policy_status"].ne("NOT_FROZEN").any()
):
    raise ValueError("800-C must not select or lose RNA source records")

print("PREFLIGHT-C source-metadata availability inventory: READY TO PERSIST")
print("Observed metadata availability is not evidence of relative sample QC")
print("Biological selection / program scoring: NOT PERFORMED")

## PREFLIGHT-D — Targeted live-GDC provenance and QC feasibility pilot

PREFLIGHT-C found no informative *probed* collection-time, specimen-localization or quantitative-QC variables in its preserved acquisition-time metadata projection. **This does not establish provider-wide absence.** The GDC data dictionary includes, for example, `samples.biospecimen_anatomic_site`, `samples.biospecimen_laterality`, `samples.portions.analytes.rna_integrity_number`, and nested read-group QC structures. Availability and suitability must be checked on exact identities, rather than inferred from the schema.

**Pilot only, before scaling to 649 files.** Select three cases each from the already observed RNA multiplicity strata `Uterus, NOS`, `Kidney`, `Larynx` (multiple samples) and `Pancreas` (multiple aliquots), by stable UUID ordering **solely for technical diagnostic sampling**. This is not a tumor-selection, quality or statistical analysis rule. No additional source file UUID is permitted. Query only those pilot file UUIDs from the public GDC `/files` endpoint and preserve a *separate*, immutable-on-rerun query snapshot. Failed API retrieval must never be interpreted as missing provider fields.

**Identity and linkage first.** Reconcile live response file UUID, filename, size and provider MD5 against Notebook 111's already audited file index, without rereading original payloads. Match any newly returned sample/portion/analyte/aliquot data to the **same exact case/sample/aliquot triplets resolved by Notebook 111**; case-level membership or presence of extra sample UUIDs is not new file-to-sample evidence. Flag discrepancies and missing nested paths explicitly. Read-group metadata can be associated with the returned file's analysis but may be absent, unlinked to the input library, or incomparable; availability does not demonstrate quality equivalence.

**Decision gate.** Count candidate fields with exact-link informative values, discrepancies, and within-case differentiation; do not select a file, impose QC thresholds, recompute sample ancestry, score any program, or automatically expand the live query to all 649 files. The next scale/no-scale decision is manual after examining this diagnostic pilot. Source release/timepoint drift is recorded as a risk; it is not silently combined with the original GDC snapshot.

In [ ]:
# =============================================================================
# Freeze a bounded, descriptive GDC pilot from the existing 288 case review
# =============================================================================

import hashlib
import time
from datetime import datetime, timezone

import requests

RNA_D_PILOT_STRATA = (
    ("MULTIPLE_SAMPLES_IN_CASE", "Uterus, NOS"),
    ("MULTIPLE_SAMPLES_IN_CASE", "Kidney"),
    ("MULTIPLE_SAMPLES_IN_CASE", "Larynx"),
    ("ONE_SAMPLE_MULTIPLE_ALIQUOTS", "Pancreas"),
)
rna_d_pilot_cases = pd.concat([
    rna_adjudication_cases.loc[
        rna_adjudication_cases["rna_multiplicity_class"].eq(kind)
        & rna_adjudication_cases["primary_site_values"].eq(site)
    ].sort_values("case_uuid", kind="stable").head(3)
    for kind, site in RNA_D_PILOT_STRATA
], ignore_index=True)

if (
    len(rna_d_pilot_cases) != 12
    or rna_d_pilot_cases["case_uuid"].duplicated().any()
):
    raise ValueError("Unexpected 12-case descriptive sampling strata")

rna_d_pilot_files = rna_adjudication_files.loc[
    rna_adjudication_files["case_uuid"].isin(
        rna_d_pilot_cases["case_uuid"]
    )
].copy()
rna_d_file_ids = sorted(rna_d_pilot_files["file_id"].tolist())
if (
    len(rna_d_file_ids) != len(set(rna_d_file_ids))
    or not set(rna_d_file_ids).issubset(
        set(rna_adjudication_files["file_id"])
    )
):
    raise ValueError("PREFLIGHT-D pilot expanded beyond Notebook 800-B file identities")

print("Pilot cases:", len(rna_d_pilot_cases))
print("Pilot exact files:", len(rna_d_file_ids))
display(rna_d_pilot_cases.groupby(
    ["rna_multiplicity_class", "primary_site_values"]
).size().rename("n_cases").reset_index())
print("Stable UUID ordering is diagnostic sampling only, NOT file eligibility")

In [ ]:
# =============================================================================
# Declare exact-file API scope and an immutable-on-rerun local source snapshot
# =============================================================================

RNA_D_GDC_URL = "https://api.gdc.cancer.gov/files"
RNA_D_API_FIELDS = [
    "file_id", "file_name", "file_size", "md5sum", "access",
    "data_type", "data_category", "proportion_reads_mapped",
    "proportion_reads_duplicated", "total_reads", "average_base_quality",
]
RNA_D_API_EXPAND = [
    "associated_entities",
    "cases.samples",
    "cases.samples.portions",
    "cases.samples.portions.analytes",
    "cases.samples.portions.analytes.aliquots",
    "analysis",
    "analysis.metadata.read_groups",
    "analysis.metadata.read_groups.read_group_qcs",
]
rna_d_query = {
    "filters": {
        "op": "in",
        "content": {"field": "file_id", "value": rna_d_file_ids},
    },
    "fields": ",".join(RNA_D_API_FIELDS),
    "expand": ",".join(RNA_D_API_EXPAND),
    "format": "JSON",
    "size": len(rna_d_file_ids),
}
rna_d_query_sha = hashlib.sha256(
    json.dumps(rna_d_query, sort_keys=True, separators=(",", ":")).encode("utf-8")
).hexdigest()

RNA_D_SNAPSHOT_PATH = OUTPUT_PATHS["rna_d_live_gdc_snapshot"]
if RNA_D_SNAPSHOT_PATH.is_file():
    with RNA_D_SNAPSHOT_PATH.open("r", encoding="utf-8") as handle:
        rna_d_snapshot = json.load(handle)
    if (
        rna_d_snapshot.get("query_sha256") != rna_d_query_sha
        or rna_d_snapshot.get("originating_111_rnaseq_metadata_sha256")
        != expected_rna_metadata_sha
    ):
        raise ValueError("Cached GDC pilot snapshot mismatches exact pilot contract")
    print("Reusing previously captured live-GDC pilot snapshot")
else:
    with requests.Session() as session:
        for attempt in range(3):
            try:
                response = session.post(
                    RNA_D_GDC_URL, json=rna_d_query, timeout=(20, 120)
                )
                response.raise_for_status()
                response_payload = response.json()
                break
            except (requests.RequestException, ValueError) as error:
                if attempt == 2:
                    raise RuntimeError(
                        "GDC pilot request failed; DO NOT infer missing fields"
                    ) from error
                time.sleep(2 ** attempt)
    data = response_payload.get("data")
    if not isinstance(data, dict) or not isinstance(data.get("hits"), list):
        raise ValueError("Unexpected GDC pilot response schema")
    hits = data["hits"]
    returned_ids = [h.get("file_id") or h.get("id") for h in hits]
    if (
        len(hits) != len(rna_d_file_ids)
        or sorted(returned_ids) != rna_d_file_ids
        or len(set(returned_ids)) != len(returned_ids)
    ):
        raise ValueError("Live GDC pilot file IDs differ from frozen review scope")
    rna_d_snapshot = {
        "source": "NCI Genomic Data Commons Files API",
        "endpoint": RNA_D_GDC_URL,
        "captured_at_utc": datetime.now(timezone.utc).isoformat(),
        "query": rna_d_query,
        "query_sha256": rna_d_query_sha,
        "originating_111_rnaseq_metadata_sha256": expected_rna_metadata_sha,
        "hits": sorted(hits, key=lambda x: x.get("file_id") or x.get("id")),
    }
    RNA_D_SNAPSHOT_PATH.parent.mkdir(parents=True, exist_ok=True)
    with RNA_D_SNAPSHOT_PATH.open("w", encoding="utf-8") as handle:
        json.dump(rna_d_snapshot, handle, indent=2, ensure_ascii=False)
        handle.write("\n")
    print("Saved separate exact-file pilot source snapshot:",
          project_relative_path(RNA_D_SNAPSHOT_PATH))

rna_d_hits = rna_d_snapshot.get("hits")
if not isinstance(rna_d_hits, list):
    raise ValueError("GDC pilot snapshot lacks hits")
rna_d_hit_by_id = {
    hit.get("file_id") or hit.get("id"): hit for hit in rna_d_hits
}
if len(rna_d_hit_by_id) != len(rna_d_file_ids) or set(
    rna_d_hit_by_id
) != set(rna_d_file_ids):
    raise ValueError("Cached pilot snapshot file IDs do not reconcile")
print("Provider snapshot SHA-256:", calculate_sha256(RNA_D_SNAPSHOT_PATH))

In [ ]:
# =============================================================================
# Reconcile live file identities and inspect EXACT pre-resolved biospecimen links
# =============================================================================

rna_d_nodes = {}
rna_d_identity_rows = []
for row in rna_d_pilot_files.itertuples(index=False):
    hit = rna_d_hit_by_id[row.file_id]
    identity_matches = (
        hit.get("file_name") == row.filename
        and str(hit.get("file_size")) == str(row.size_bytes)
        and str(hit.get("md5sum", "")).lower() == str(row.provider_md5).lower()
    )
    if not identity_matches:
        raise ValueError("GDC live file identity differs from Notebook 111: "
                         + str(row.file_id))

    direct = hit.get("associated_entities")
    if not isinstance(direct, list):
        direct_status = "UNAVAILABLE_IN_LIVE_RESPONSE"
    else:
        direct_links = [
            entity for entity in direct
            if entity.get("entity_type") == "aliquot"
        ]
        exact = [
            entity for entity in direct_links
            if entity.get("entity_id") == row.aliquot_uuid
            and entity.get("case_id") == row.case_uuid
        ]
        direct_status = (
            "EXACT_MATCH" if len(direct_links) == len(exact) == 1
            else "DIRECT_ASSOCIATION_DISAGREEMENT"
        )

    paths = exact_source_nodes(
        hit, row.case_uuid, row.sample_uuid, row.aliquot_uuid
    )
    # The LIVE case-scoped expansion is never used to reassign ancestry.
    if paths:
        node_map = {
            scope: [entry[scope] for entry in paths]
            for scope in ("sample", "portion", "analyte", "aliquot")
        }
    else:
        node_map = {scope: [] for scope in (
            "sample", "portion", "analyte", "aliquot"
        )}

    analysis = hit.get("analysis")
    read_groups = ((analysis.get("metadata") or {}).get("read_groups") or []) if (
        isinstance(analysis, dict)
        and isinstance(analysis.get("metadata"), dict)
    ) else []
    if not isinstance(read_groups, list):
        read_groups = []
    qcs = [
        qc for group in read_groups if isinstance(group, dict)
        for qc in (group.get("read_group_qcs") or [])
        if isinstance(qc, dict)
    ]
    node_map.update({
        "file": [hit],
        "analysis": [analysis] if isinstance(analysis, dict) else [],
        "read_group_qc": qcs,
    })
    rna_d_nodes[row.file_id] = node_map
    rna_d_identity_rows.append({
        "file_id": row.file_id, "case_uuid": row.case_uuid,
        "sample_uuid": row.sample_uuid, "aliquot_uuid": row.aliquot_uuid,
        "rna_multiplicity_class": row.rna_multiplicity_class,
        "primary_site": row.primary_site,
        "live_file_identity": "EXACT_MATCH",
        "live_direct_association": direct_status,
        "n_exact_hierarchy_paths": len(paths),
        "n_read_groups": len(read_groups),
        "n_read_group_qcs": len(qcs),
    })

rna_d_identity = pd.DataFrame(rna_d_identity_rows).sort_values(
    ["case_uuid", "file_id"]
).reset_index(drop=True)
display(rna_d_identity.groupby(
    ["primary_site", "live_direct_association"]
).agg(
    n_files=("file_id", "nunique"),
    n_files_with_exact_nested_path=(
        "n_exact_hierarchy_paths", lambda s: int(s.gt(0).sum())
    ),
    n_files_with_qc_records=(
        "n_read_group_qcs", lambda s: int(s.gt(0).sum())
    ),
).reset_index())
print("No live case-scoped linkage is accepted as a replacement for Notebook 111")

In [ ]:
# =============================================================================
# Audit new provider fields and nested QC availability without quality ranking
# =============================================================================

RNA_D_PROBES = {
    "spatial_specimen": [
        ("sample", "biospecimen_anatomic_site"),
        ("sample", "biospecimen_laterality"),
        ("sample", "tissue_collection_type"),
        ("sample", "specimen_type"),
    ],
    "collection_provenance": [
        ("sample", "days_to_collection"),
        ("sample", "days_to_sample_procurement"),
        ("sample", "method_of_sample_procurement"),
        ("sample", "preservation_method"),
        ("sample", "freezing_method"),
        ("sample", "time_between_excision_and_freezing"),
    ],
    "molecular_preparation": [
        ("portion", "is_ffpe"), ("portion", "portion_number"),
        ("analyte", "analyte_type"),
        ("analyte", "rna_integrity_number"),
        ("analyte", "ribosomal_rna_28s_18s_ratio"),
        ("analyte", "a260_a280_ratio"),
        ("analyte", "concentration"),
        ("aliquot", "source_center"),
    ],
    "file_level_qc_probes": [
        ("file", "proportion_reads_mapped"),
        ("file", "proportion_reads_duplicated"),
        ("file", "total_reads"),
        ("file", "average_base_quality"),
    ],
    "read_group_qc_probes": [
        ("read_group_qc", "total_sequences"),
        ("read_group_qc", "percent_gc_content"),
        ("read_group_qc", "basic_statistics"),
    ],
}

rna_d_field_rows = []
for file_id, node_map in rna_d_nodes.items():
    identity = rna_d_identity.loc[
        rna_d_identity["file_id"].eq(file_id)
    ].iloc[0]
    linked = bool(
        identity["n_exact_hierarchy_paths"] > 0
        and identity["live_direct_association"] == "EXACT_MATCH"
    )
    for domain, probes in RNA_D_PROBES.items():
        for scope, field in probes:
            nodes = node_map[scope] if (
                linked or scope in ("file", "read_group_qc")
            ) else []
            present, informative, discordant = source_value_status(nodes, field)
            # Structured FastQC summaries are reported as structural presence;
            # their contents are NOT converted to quantitative QC scores.
            structured = any(
                isinstance(node.get(field), (dict, list))
                for node in nodes if isinstance(node, dict)
            )
            values = sorted({
                str(node[field]).strip()
                for node in nodes if isinstance(node, dict)
                and field in node and informative_scalar(node[field])
            })
            rna_d_field_rows.append({
                "file_id": file_id,
                "case_uuid": identity["case_uuid"],
                "sample_uuid": identity["sample_uuid"],
                "aliquot_uuid": identity["aliquot_uuid"],
                "primary_site": identity["primary_site"],
                "domain": domain,
                "scope": scope,
                "field": field,
                "biospecimen_link_eligible_for_evidence": linked,
                "key_present": present,
                "informative_scalar": informative,
                "informative_values_json": json.dumps(values, ensure_ascii=False),
                "structured_value_present": structured,
                "multiple_informative_values_within_file": discordant,
                "interpretation": (
                    "AVAILABILITY_ONLY_NOT_A_QC_THRESHOLD"
                    if scope in ("file", "read_group_qc")
                    else "EXACT_LINKED_BIOSPECIMEN_METADATA_ONLY"
                ),
            })
rna_d_field_evidence = pd.DataFrame(rna_d_field_rows).sort_values(
    ["case_uuid", "file_id", "domain", "field"], kind="stable"
).reset_index(drop=True)
display(rna_d_field_evidence.groupby(["domain", "field"]).agg(
    n_files_with_informative_scalar=("informative_scalar", "sum"),
    n_files_with_structured_field=("structured_value_present", "sum"),
    n_files_with_key=("key_present", "sum"),
).reset_index())
print("No field value interpreted as sufficient for a selection rule")

In [ ]:
# =============================================================================
# Quantify whether available attributes differ across files within each case
# =============================================================================

rna_d_case_field_comparison = (
    rna_d_field_evidence.groupby(["case_uuid", "domain", "field"])
    .agg(
        n_files_with_key=("key_present", "sum"),
        n_files_informative=("informative_scalar", "sum"),
        n_files_structured=("structured_value_present", "sum"),
        n_files_without_exact_link=(
            "biospecimen_link_eligible_for_evidence",
            lambda values: int((~values).sum())
        ),
        n_distinct_informative_values=(
            "informative_values_json",
            lambda values: len({
                item for raw in values for item in json.loads(raw)
            }),
        ),
    ).reset_index()
)
rna_d_case_field_comparison["candidate_values_differ"] = (
    rna_d_case_field_comparison["n_distinct_informative_values"].gt(1)
)
display(
    rna_d_case_field_comparison.groupby("domain")
    .agg(
        n_case_fields_informative=(
            "n_files_informative", lambda v: int(v.gt(0).sum())
        ),
        n_case_fields_with_distinct_values=("candidate_values_differ", "sum"),
    ).reset_index()
)

rna_d_case_summary = (
    rna_d_pilot_cases[[
        "case_uuid", "primary_site_values", "rna_multiplicity_class",
        "n_files", "n_samples", "n_aliquots",
    ]].merge(
        rna_d_identity.groupby("case_uuid")
        .agg(
            n_files_with_matching_file_identity=("live_file_identity", "size"),
            n_files_with_exact_biospecimen_link=(
                "live_direct_association",
                lambda s: int(s.eq("EXACT_MATCH").sum()),
            ),
            n_files_with_nested_path=(
                "n_exact_hierarchy_paths", lambda s: int(s.gt(0).sum())
            ),
            n_files_with_read_group_qc=(
                "n_read_group_qcs", lambda s: int(s.gt(0).sum())
            ),
        ).reset_index(),
        on="case_uuid", validate="one_to_one"
    )
)
for domain in RNA_D_PROBES:
    coverage = (
        rna_d_field_evidence.loc[
            rna_d_field_evidence["domain"].eq(domain)
            & rna_d_field_evidence["informative_scalar"]
        ]
        .groupby("case_uuid")["file_id"].nunique()
    )
    rna_d_case_summary["files_informative_" + domain] = (
        rna_d_case_summary["case_uuid"].map(coverage).fillna(0).astype("int64")
    )
rna_d_case_summary["selection_status"] = "NOT_FROZEN"
display(rna_d_case_summary.groupby(
    ["rna_multiplicity_class", "primary_site_values"]
).agg(
    n_cases=("case_uuid", "nunique"),
    n_files=("n_files", "sum"),
    exact_nested_path_files=("n_files_with_nested_path", "sum"),
    read_group_qc_files=("n_files_with_read_group_qc", "sum"),
    files_with_spatial_specimen=("files_informative_spatial_specimen", "sum"),
    files_with_collection_provenance=(
        "files_informative_collection_provenance", "sum"
    ),
    files_with_molecular_preparation=(
        "files_informative_molecular_preparation", "sum"
    ),
    files_with_file_qc_probes=(
        "files_informative_file_level_qc_probes", "sum"
    ),
).reset_index())
print("Full 649-file recovery feasibility: NOT ASSESSED — pilot only")

In [ ]:
# =============================================================================
# Preserve no-adjudication and precise scope before writing pilot diagnostics
# =============================================================================

if (
    len(rna_d_case_summary) != 12
    or rna_d_case_summary["case_uuid"].duplicated().any()
    or len(rna_d_identity) != len(rna_d_file_ids)
    or len(rna_d_field_evidence) != (
        len(rna_d_file_ids) * sum(len(p) for p in RNA_D_PROBES.values())
    )
    or not set(rna_d_identity["file_id"]).issubset(
        set(rna_adjudication_files["file_id"])
    )
    or rna_d_case_summary["selection_status"].ne("NOT_FROZEN").any()
    or rna_adjudication_files["candidate_selected"].any()
    or rna_adjudication_cases["selected_file_id"].notna().any()
):
    raise ValueError("PREFLIGHT-D violated exact scope or no-selection boundary")

print("PREFLIGHT-D pilot diagnostic ready for review; no automatic expansion")
print("All 288 multiplicity cases remain unresolved, no biological scoring")

## Lexical identifier-overlap audit

Compare exact GDC case UUID and submitter strings against the frozen TCGA cohort. This is a **collision screen only**. Separate projects may assign different IDs to the same biological donor; therefore an absence of collisions is not evidence of documented donor independence. A provenance/crosswalk review remains a gate before confirmatory replication claims.

In [ ]:
# =============================================================================
# Read TCGA source identities from the frozen Phase 2 sample mapping
# =============================================================================

rna_primary = candidate_primary.loc[
    candidate_primary["modality"].eq("rnaseq")
].copy()

tcga_mapping = pd.read_csv(frozen_paths["tcga_sample_mapping"], dtype="string")
tcga_required = {"rna_case_uuid", "case_submitter_id"}
if tcga_required - set(tcga_mapping.columns):
    raise KeyError("Frozen TCGA mapping lacks required case identifiers")

tcga_case_uuid_set = set(tcga_mapping["rna_case_uuid"].dropna())
tcga_submitter_set = set(tcga_mapping["case_submitter_id"].dropna())

print("Candidate CPTAC-3 RNA cases:", rna_primary["case_uuid"].nunique())
print("Donor independence: UNRESOLVED without source crosswalk")

In [ ]:
# =============================================================================
# Create a case-level exact-string collision screen, not independence labels
# =============================================================================

identifier_collisions = (
    candidate_primary.groupby(["modality", "case_uuid"], dropna=False)
    .agg(
        n_submitter_ids=("case_submitter_id", "nunique"),
        submitter_id_values=(
            "case_submitter_id",
            lambda v: "|".join(sorted(set(v.dropna()))),
        ),
        primary_site_values=(
            "primary_site", lambda v: "|".join(sorted(set(v.dropna())))
        ),
        disease_type_values=(
            "disease_type", lambda v: "|".join(sorted(set(v.dropna())))
        ),
    )
    .reset_index()
)
identifier_collisions["tcga_case_uuid_collision"] = (
    identifier_collisions["case_uuid"].isin(tcga_case_uuid_set)
)
identifier_collisions["tcga_submitter_collision"] = (
    identifier_collisions["submitter_id_values"].map(
        lambda s: any(v in tcga_submitter_set for v in s.split("|") if v)
    )
)
identifier_collisions["donor_independence_status"] = (
    "UNRESOLVED_WITHOUT_EXTERNAL_PROVENANCE"
)
identifier_collisions["case_submitter_ambiguity"] = (
    identifier_collisions["n_submitter_ids"].gt(1)
)

display(
    identifier_collisions.groupby("modality")
    .agg(
        candidate_cases=("case_uuid", "nunique"),
        exact_uuid_collision_rows=("tcga_case_uuid_collision", "sum"),
        exact_submitter_collision_rows=("tcga_submitter_collision", "sum"),
    )
)

## Pilot technical gene-ID compatibility

The source library has already passed Notebook 111 file-integrity checks. A single deterministically selected *provisional primary RNA-seq* file is opened here **only** to inspect the GDC STAR Counts header and Ensembl identifier mapping to the 2,389-gene frozen universe.

This is a technical pilot, **not** a complete feature-coverage assessment, cohort normalization, or program test. A match after stripping Ensembl versions is reported only as a diagnostic; no version stripping, symbol fallback, missing-gene imputation or rescue is applied. Full per-file/lineage compatibility and thresholds remain open until reviewed.

In [ ]:
# =============================================================================
# Select one deterministic RNA-seq technical pilot from the frozen source files
# =============================================================================

if rna_primary.empty:
    raise ValueError("Notebook 111 has no provisional primary RNA-seq candidates")
pilot_record = rna_primary.sort_values(
    ["file_id", "filename"], kind="stable"
).iloc[0]
pilot_file_uuid = pilot_record["file_id"]
pilot_path = (
    Paths.root / "data" / "raw" / "cptac3" / "rnaseq_star_counts"
    / pilot_file_uuid / pilot_record["filename"]
)
if not pilot_path.is_file():
    raise FileNotFoundError(f"Frozen Notebook 111 pilot payload is missing: {pilot_path}")

pilot_header = pd.read_csv(pilot_path, sep="\t", comment="#", nrows=0).columns.tolist()
print("Pilot GDC file:", pilot_file_uuid)
print("STAR Counts fields:", pilot_header)
if not {"gene_id", "unstranded"}.issubset(pilot_header):
    raise ValueError("Unexpected STAR Counts schema; no gene crosswalk attempted")

In [ ]:
# =============================================================================
# Audit exact/version-insensitive Ensembl ID coverage in the technical pilot
# =============================================================================

pilot_gene_ids = pd.read_csv(
    pilot_path, sep="\t", comment="#", usecols=["gene_id"], dtype="string"
)["gene_id"].dropna()
provider_ids = set(pilot_gene_ids.tolist())
provider_versions_stripped = pilot_gene_ids.str.replace(
    r"\.\d+$", "", regex=True
)
base_id_counts = provider_versions_stripped.value_counts()
ambiguous_base_ids = set(base_id_counts[base_id_counts.gt(1)].index)

gene_id_pilot = shared_genes[
    ["gene_symbol", "tumor_gene_id", "cell_line_gene_id"]
].copy()
gene_id_pilot["pilot_file_id"] = pilot_file_uuid
gene_id_pilot["exact_ensembl_match"] = (
    gene_id_pilot["tumor_gene_id"].isin(provider_ids)
)
gene_id_pilot["versionless_pilot_match_diagnostic"] = (
    gene_id_pilot["tumor_gene_id"].str.replace(r"\.\d+$", "", regex=True)
    .isin(set(provider_versions_stripped))
)
gene_id_pilot["versionless_pilot_id_ambiguous"] = (
    gene_id_pilot["tumor_gene_id"].str.replace(r"\.\d+$", "", regex=True)
    .isin(ambiguous_base_ids)
)
gene_id_pilot["mapping_applied"] = False

display(gene_id_pilot[
    ["exact_ensembl_match", "versionless_pilot_match_diagnostic",
     "versionless_pilot_id_ambiguous"]
].sum().to_frame(name="genes"))
print("Cohort-wide feature compatibility: NOT ASSESSED")

## Readiness and persistence

The preflight outputs remain local, provisional diagnostics under `data/interim/metadata`. They must not be registered as frozen biological validation evidence or used for Phase 9. The RNA-seq per-case selection and lineage crosswalk remain unresolved; project-specific donor independence remains unverified; the statistical Phase 8 analysis contract is not frozen.

New outputs receive basic shape/column validation after persistence, while Notebook 111's prior raw MD5 scan, upstream source inventories and frozen program-construction logic are not rerun.

In [ ]:
# =============================================================================
# Assemble explicit readiness statuses and input provenance
# =============================================================================

input_provenance = {
    "111_summary": {
        "path": project_relative_path(AUDIT_SUMMARY_PATH),
        "sha256": calculate_sha256(AUDIT_SUMMARY_PATH),
    },
    "111_registry_proposal": {
        "path": project_relative_path(RAW_REGISTRY_PROPOSAL_PATH),
        "sha256": calculate_sha256(RAW_REGISTRY_PROPOSAL_PATH),
    },
}
input_provenance.update({
    "111_" + name: {
        "path": project_relative_path(path),
        "sha256": audit_summary["audit_artifacts"][name]["sha256"],
    }
    for name, path in audit_inputs.items()
})
input_provenance["gdc_targeted_rna_pilot_for_d"] = {
    "path": project_relative_path(RNA_D_SNAPSHOT_PATH),
    "sha256": calculate_sha256(RNA_D_SNAPSHOT_PATH),
    "query_sha256": rna_d_query_sha,
    "captured_at_utc": rna_d_snapshot.get("captured_at_utc"),
}
input_provenance["gdc_rna_acquisition_metadata_for_c"] = {
    "path": project_relative_path(RNA_GDC_METADATA_PATH),
    "sha256": expected_rna_metadata_sha,
}
input_provenance.update({
    artifact_id: {
        "path": get_artifact(artifact_registry, artifact_id)["path"],
        "sha256": get_artifact(artifact_registry, artifact_id)["sha256"],
    }
    for artifact_id in FROZEN_ARTIFACT_IDS.values()
})

readiness_status = {
    "notebook111_source_integrity": "UPSTREAM_PASS_REUSED",
    "notebook111_local_handoff_identity": "PASS",
    "phase4_frozen_identity": "PASS",
    "provisional_primary_inventory": "AVAILABLE_NOT_COHORT_FROZEN",
    "one_record_per_case": "PENDING_POLICY",
    "rna_multiplicity_adjudication_evidence": "DESCRIBED_UNRESOLVED_NO_SELECTION",
    "rna_specimen_provenance_metadata": "SNAPSHOT_AVAILABILITY_AUDITED",
    "rna_quantitative_qc_metadata": "SNAPSHOT_PROBES_ONLY_NO_PAYLOAD_QC",
    "rna_one_record_per_case_selection": "NOT_FROZEN",
    "rna_live_gdc_pilot": "TARGETED_SOURCE_SNAPSHOT_CAPTURED_NOT_COHORT_WIDE",
    "rna_d_additional_provenance_qc": "PILOT_AVAILABILITY_REVIEW_PENDING",
    "rna_case_multiplicity_classes": "DESCRIPTIVE_COMPLETE_NOT_RESOLVED",
    "rna_cross_file_ancestry_conflicts": (
        "REVIEW_REQUIRED" if rna_cases["cross_file_ancestry_conflict"].any()
        else "NONE_DETECTED_WITHIN_PROVISIONAL_PRIMARY_FILES"
    ),
    "harmonized_lineage_crosswalk": "NOT_ASSESSED",
    "donor_independence": "UNRESOLVED",
    "star_counts_gene_compatibility": "PILOT_ONLY",
    "full_transcriptomic_qc_and_coverage": "NOT_ASSESSED",
    "methylation_platform_and_sample_harmonization": "NOT_ASSESSED",
    "external_replication_analysis_contract": "NOT_FROZEN",
    "biological_validation_authorized": False,
}
display(pd.DataFrame(
    [{"gate": gate, "status": status} for gate, status in readiness_status.items()]
))

In [ ]:
# =============================================================================
# Persist new technical inventories; leave registries and raw data untouched
# =============================================================================

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_tables = {
    "lineage_inventory": lineage_inventory,
    "case_multiplicity": case_multiplicity,
    "identifier_collisions": identifier_collisions,
    "gene_id_pilot": gene_id_pilot,
    "rna_adjudication_case_evidence": rna_adjudication_cases,
    "rna_adjudication_file_evidence": rna_adjudication_files,
    "rna_c_field_coverage": rna_c_field_coverage,
    "rna_c_file_availability": rna_c_file_availability,
    "rna_c_case_coverage": rna_c_case_coverage,
    "rna_d_identity": rna_d_identity,
    "rna_d_field_evidence": rna_d_field_evidence,
    "rna_d_case_summary": rna_d_case_summary,
    "rna_d_case_field_comparison": rna_d_case_field_comparison,
}
for name, frame in output_tables.items():
    frame.to_csv(OUTPUT_PATHS[name], index=False)
    print(name, "→", project_relative_path(OUTPUT_PATHS[name]), "rows:", len(frame))

In [ ]:
# =============================================================================
# Validate newly persisted technical inventories, not all upstream schemas
# =============================================================================

for name, frame in output_tables.items():
    reloaded = pd.read_csv(OUTPUT_PATHS[name], dtype="string")
    if len(reloaded) != len(frame) or list(reloaded.columns) != list(frame.columns):
        raise ValueError(f"New preflight output is inconsistent: {name}")

print("New technical inventory persistence: PASS")

In [ ]:
# =============================================================================
# Write the non-frozen readiness record with provenance and output fingerprints
# =============================================================================

readiness_record = {
    "notebook": "800_external_bulk_cohorts",
    "stage": "PREFLIGHT_A_B_C_AND_D_TARGETED_PILOT_ONLY",
    "environment": {
        "python": platform.python_version(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
    },
    "upstream_111_snapshot": audit_summary["source_fingerprints"],
    "inputs": input_provenance,
    "raw_registry_cptac3_present": cptac3_registered,
    "source_count_context": audit_summary["n_files_by_modality"],
    "program_gene_universe_n": len(shared_genes),
    "rna_case_multiplicity_class_counts": {
        str(row.rna_multiplicity_class): int(row.n_cases)
        for row in rna_class_summary.itertuples(index=False)
    },
    "rna_preflight_b_evidence_class_counts": {
        str(k): int(v) for k, v in
        rna_adjudication_cases["adjudication_evidence_class"]
        .value_counts(dropna=False).items()
    },
    "rna_preflight_b_case_count": len(rna_adjudication_cases),
    "rna_preflight_b_descriptor_discordance_cases": int(
        rna_adjudication_cases["source_descriptor_discordance"].sum()
    ),
    "rna_preflight_b_descriptor_incomplete_cases": int(
        rna_adjudication_cases["source_descriptor_incomplete"].sum()
    ),
    "rna_preflight_b_selection_performed": False,
    "rna_preflight_c_reviewed_files": len(rna_c_file_availability),
    "rna_preflight_c_reviewed_cases": len(rna_c_case_coverage),
    "rna_preflight_c_quantitative_qc_snapshot_files": int(
        rna_c_file_availability[
            "informative_quantitative_qc_metadata_probes"
        ].sum()
    ),
    "rna_preflight_c_selection_performed": False,
    "rna_preflight_d_pilot_cases": len(rna_d_case_summary),
    "rna_preflight_d_pilot_files": len(rna_d_identity),
    "rna_preflight_d_case_fields_with_distinct_values": int(
        rna_d_case_field_comparison["candidate_values_differ"].sum()
    ),
    "rna_preflight_d_gdc_snapshot_sha256": calculate_sha256(
        RNA_D_SNAPSHOT_PATH
    ),
    "rna_preflight_d_pilot_selection_performed": False,
    "rna_preflight_d_full_649_expansion_performed": False,
    "pilot_file_uuid": str(pilot_file_uuid),
    "pilot_star_counts_fields": pilot_header,
    "readiness": readiness_status,
    "cohort_frozen": False,
    "program_scores_computed": False,
    "external_replication_test_performed": False,
    "outputs": {
        name: {
            "path": project_relative_path(OUTPUT_PATHS[name]),
            "rows": len(output_tables[name]),
            "sha256": calculate_sha256(OUTPUT_PATHS[name]),
        }
        for name in output_tables
    },
}
with OUTPUT_PATHS["readiness"].open("w", encoding="utf-8") as handle:
    json.dump(readiness_record, handle, indent=2, ensure_ascii=False)
    handle.write("\n")
with OUTPUT_PATHS["readiness"].open("r", encoding="utf-8") as handle:
    persisted_readiness = json.load(handle)
if persisted_readiness["readiness"] != readiness_status:
    raise ValueError("Persisted readiness differs from the in-memory record")
print("Readiness saved:", project_relative_path(OUTPUT_PATHS["readiness"]))

## PREFLIGHT-A–C — executed technical checkpoint (cohort NOT frozen)

**Scope and execution.** Notebook 800 completed the outcome-blind CPTAC-3 technical preflight through **PREFLIGHT-C**, with **35/35 code cells executed without errors** in the recorded local run. It reuses Notebook 111's acquisition audit and verified handoff fingerprints, and checks the required frozen Phase 2/4 input identities. It does **not** rerun the full GDC acquisition/integrity audit, alter upstream frozen programs, compute program scores, or test external biological replication. CPTAC-3 remains **not registered as an acquired-and-used analytic cohort**.

### PREFLIGHT-A — source inventory and case multiplicity

The provisional RNA primary-tumor inventory comprises **1,370 distinct cases**: **1,082** with one file/sample/aliquot, **276** with multiple sample UUIDs per case, and **12** with multiple aliquots of one sample. No conflicting cross-file sample/aliquot ancestry was detected *within this provisional uniquely resolved primary-tumor subset*. This does not resolve the **314 RNA files with ambiguous sample ancestry** identified in Notebook 111. The deterministic STAR Counts pilot exactly matched **2,389/2,389** frozen gene IDs in **one** file; cohort-wide coverage and preprocessing compatibility remain unassessed. Exact-string collision checks against the frozen TCGA mapping returned zero matches but **do not establish independent donors**.

### PREFLIGHT-B — source-descriptor evidence (accepted; selection unresolved)

The **288** cases requiring multiplicity review contain **649** provisionally mapped RNA files (**619** in the multiple-sample group; **30** in the multiple-aliquot group). The audited file/sample/aliquot descriptors examined in B were present, and no within-case disagreement was detected for the checked source labels. Their agreement is **not** proof of biological equivalence, shared sampling region/timepoint, technical replicate status, or comparable read quality. **No file or sample was selected, averaged, weighted, or excluded.**

### PREFLIGHT-C — acquisition-snapshot provenance and QC availability (executed)

The preserved GDC RNA metadata snapshot matched Notebook 111's recorded SHA-256, and all **649** reviewed file UUIDs had an exact match to their already audited case/sample/aliquot hierarchy triplet (**0** files with multiple exact-path rows). The availability inventory contains **50** scoped scalar-field/probe rows, including **39** prespecified field probes and **11** additional observed scalar fields.

Within this **specific acquisition-time metadata projection**:

- Generic sample descriptors, administrative submitter IDs, and workflow identifiers/versions were informative for **649/649 files**. These are **not** specific spatial, temporal, or quantitative QC evidence.
- The prespecified **collection-time, sampling-location, specimen-processing, and quantitative sequencing-QC probes** had **0/649 informative file values**. Consequently **0/288 cases** had a file with informative probed collection timing, sampling-location detail, or quantitative QC metadata.
- No prospective specimen-selection preference can be established from the audited fields.

**Interpretation boundary.** These zeros mean that the particular fields were not informative **in the preserved snapshot**; they do **not** mean that GDC or another provider has no additional biospecimen provenance or QC. The 11 extra observed scalar fields are inventoried, not independently established as suitable adjudication variables. STAR payload-derived QC, original sequencing/run reports, potential spatial or longitudinal sampling differences, and relative quality **were not evaluated**.

### Readiness and next decision

**PREFLIGHT-A, -B and -C have executed as technical evidence inventories; none constitutes cohort adjudication or biological validation.** The nine local CSV inventories and the readiness JSON are **non-frozen** diagnostic outputs. `one_record_per_case=PENDING_POLICY`, donor independence is **UNRESOLVED**, lineage harmonization and cohort-wide RNA compatibility are **NOT ASSESSED**, and the prospective external replication contract is **NOT FROZEN**. `biological_validation_authorized=false`.

**PREFLIGHT-D targeted pilot added (NOT YET INTERPRETED).** A bounded diagnostic set of 12 cases from the already audited 288-case RNA multiplicity subset is queried via exact file UUIDs from the live GDC Files API, without increasing the frozen acquisition universe. GDC file identity is checked against Notebook 111; any newly returned specimen data are restricted to the pre-resolved exact sample/aliquot links. Fields and read-group QC are audited for availability only; no QC ranking or case adjudication is performed, and no automatic expansion to 649 files is allowed. The returned query snapshot is preserved separately and fingerprinted, without replacing the 111 acquisition snapshot. Its *observed* provenance and quality-field utility must be reviewed after execution.

**Next task:** decide whether *independent, file-/sample-linked* specimen provenance and objective comparable RNA QC can be obtained without changing the frozen acquisition universe. If not, preserve the **288 multiplicity cases as unresolved**; formulate and assess a prospectively specified, lineage-aware eligibility/sensitivity strategy (including the 1,082 structurally simple cases as a candidate subset, **not** automatically eligible). Address donor independence, RNA preprocessing and full feature coverage, and an independent replication estimand with matched nulls **before** authorizing any scores or external program-replication inference.